<a href="https://colab.research.google.com/github/niofrequency/google-collab_image/blob/main/QWEN-2511-IMAGE-EDIT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI /content/ComfyUI
!cd /content/ComfyUI && pip install -q -r requirements.txt
!pip install -q torchsde

!apt install aria2 -qqy

# Create folders
!mkdir -p /content/ComfyUI/models/diffusion_models
!mkdir -p /content/ComfyUI/models/text_encoders
!mkdir -p /content/ComfyUI/models/vae
!mkdir -p /content/ComfyUI/models/loras

# Download Diffusion model (FP8 mixed ~20.5GB - recommended)
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  https://huggingface.co/Comfy-Org/Qwen-Image-Edit_ComfyUI/resolve/main/split_files/diffusion_models/qwen_image_edit_2511_fp8mixed.safetensors \
  -d /content/ComfyUI/models/diffusion_models -o qwen_image_edit_2511_fp8mixed.safetensors

# Download Text Encoder
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors \
  -d /content/ComfyUI/models/text_encoders -o qwen_2.5_vl_7b_fp8_scaled.safetensors

# Download VAE
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/vae/qwen_image_vae.safetensors \
  -d /content/ComfyUI/models/vae -o qwen_image_vae.safetensors

# Download Lightning LoRA (optional, for 4-step speed)
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  https://huggingface.co/lightx2v/Qwen-Image-Edit-2511-Lightning/resolve/main/Qwen-Image-Edit-2511-Lightning-4steps-V1.0-bf16.safetensors \
  -d /content/ComfyUI/models/loras -o Qwen-Image-Edit-2511-Lightning-4steps-V1.0-bf16.safetensors

# Check downloads
!ls -lh /content/ComfyUI/models/diffusion_models/
!ls -lh /content/ComfyUI/models/text_encoders/
!ls -lh /content/ComfyUI/models/vae/
!ls -lh /content/ComfyUI/models/loras/

/content
fatal: destination path '/content/ComfyUI' already exists and is not an empty directory.
aria2 is already the newest version (1.37.0+debian-1build3).
0 upgraded, 0 newly installed, 0 to remove and 33 not upgraded.
 *** Download Progress Summary as of Wed Nov 11 03:24:45 2026 *** 
=
[#fe363b 11GiB/19GiB(61%) CN:16 DL:181MiB ETA:41s]
FILE: /content/ComfyUI/models/diffusion_models/qwen_image_edit_2511_fp8mixed.safetensors
-


Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
fe363b|OK  |   174MiB/s|/content/ComfyUI/models/diffusion_models/qwen_image_edit_2511_fp8mixed.safetensors

Status Legend:
(OK):download completed.
 *** Download Progress Summary as of Wed Nov 11 03:26:38 2026 *** 
=
[#0af5ca 8.7GiB/8.7GiB(99%) CN:1 DL:138MiB]
FILE: /content/ComfyUI/models/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors
-


Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+===========

In [ ]:
%cd /content/ComfyUI

import torch
import logging
import warnings

logging.getLogger("root").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

from nodes import NODE_CLASS_MAPPINGS

UNETLoader = NODE_CLASS_MAPPINGS["UNETLoader"]()
CLIPLoader = NODE_CLASS_MAPPINGS["CLIPLoader"]()
VAELoader = NODE_CLASS_MAPPINGS["VAELoader"]()
LoraLoaderModelOnly = NODE_CLASS_MAPPINGS["LoraLoaderModelOnly"]()

with torch.inference_mode():
    # Load UNET
    unet = UNETLoader.load_unet("qwen_image_edit_2511_fp8mixed.safetensors", "default")[0]

    # Load CLIP (text encoder)
    clip = CLIPLoader.load_clip("qwen_2.5_vl_7b_fp8_scaled.safetensors", "qwen_image")[0]

    # Load VAE
    vae = VAELoader.load_vae("qwen_image_vae.safetensors")[0]

    # Optional: Apply Lightning LoRA for 4-step speed
    # unet = LoraLoaderModelOnly.load_lora_model_only(unet, "Qwen-Image-Edit-2511-Lightning-4steps-V1.0-bf16.safetensors", 1.0)[0]

print("Qwen-Image-Edit-2511 loaded successfully.")

In [ ]:
%cd /content/ComfyUI

import torch
import random
from PIL import Image
from IPython.display import display
from nodes import NODE_CLASS_MAPPINGS

# Ensure models are loaded in global scope
if 'unet' not in globals() or 'clip' not in globals() or 'vae' not in globals():
    print("Loading models...")
    UNETLoader = NODE_CLASS_MAPPINGS["UNETLoader"]()
    CLIPLoader = NODE_CLASS_MAPPINGS["CLIPLoader"]()
    VAELoader = NODE_CLASS_MAPPINGS["VAELoader"]()

    with torch.inference_mode():
        unet = UNETLoader.load_unet("qwen_image_edit_2511_fp8mixed.safetensors", "default")[0]
        clip = CLIPLoader.load_clip("qwen_2.5_vl_7b_fp8_scaled.safetensors", "qwen_image")[0]
        vae = VAELoader.load_vae("qwen_image_vae.safetensors")[0]
    print("Models loaded.")

CLIPTextEncode = NODE_CLASS_MAPPINGS["CLIPTextEncode"]()
ConditioningZeroOut = NODE_CLASS_MAPPINGS["ConditioningZeroOut"]()
EmptyLatentImage = NODE_CLASS_MAPPINGS["EmptyLatentImage"]()
KSampler = NODE_CLASS_MAPPINGS["KSampler"]()
VAEDecode = NODE_CLASS_MAPPINGS["VAEDecode"]()

positive = "photorealistic portrait of a woman, natural skin texture, detailed face, high quality"
negative = ""

with torch.inference_mode():
    pos = CLIPTextEncode.encode(clip, positive)[0]
    neg = ConditioningZeroOut.zero_out(pos)[0]
    latent = EmptyLatentImage.generate(1024, 1024, 1)[0]

    samples = KSampler.sample(
        unet,
        random.randint(1, 2**32),
        20,           # steps
        4.0,          # cfg
        "euler",
        "simple",
        pos,
        neg,
        latent,
        1.0
    )[0]

    image = VAEDecode.decode(vae, samples)[0]
    img = Image.fromarray((image.cpu().numpy().clip(0,1)*255).astype("uint8"))
    display(img)

/content/ComfyUI
Loading models...
Models loaded.


In [ ]:
%cd /content/ComfyUI

import threading, time, subprocess, re, base64, io, random, torch, gc
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn
from PIL import Image
import numpy as np
from nodes import NODE_CLASS_MAPPINGS
import logging, warnings
import requests

NGROK_AUTHTOKEN = "3Lwf9MDe4QNmfXZFcPk4JozeOfi_4uv8uyfawsHGLuVYhMN6Q"

logging.getLogger("root").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

print("Loading models...")
UNETLoader = NODE_CLASS_MAPPINGS["UNETLoader"]()
CLIPLoader = NODE_CLASS_MAPPINGS["CLIPLoader"]()
VAELoader = NODE_CLASS_MAPPINGS["VAELoader"]()

with torch.inference_mode():
    unet = UNETLoader.load_unet("qwen_image_edit_2511_fp8mixed.safetensors", "default")[0]
    clip = CLIPLoader.load_clip("qwen_2.5_vl_7b_fp8_scaled.safetensors", "qwen_image")[0]
    vae = VAELoader.load_vae("qwen_image_vae.safetensors")[0]
print("Models loaded.")

CLIPTextEncode = NODE_CLASS_MAPPINGS["CLIPTextEncode"]()
ConditioningZeroOut = NODE_CLASS_MAPPINGS["ConditioningZeroOut"]()
EmptyLatentImage = NODE_CLASS_MAPPINGS["EmptyLatentImage"]()
KSampler = NODE_CLASS_MAPPINGS["KSampler"]()
VAEDecode = NODE_CLASS_MAPPINGS["VAEDecode"]()

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class GenerateRequest(BaseModel):
    positive: str = ""
    negative: str = ""
    steps: int = 20
    cfg: float = 4.0
    width: int = 1024
    height: int = 1024
    seed: int = -1
    sampler: str = "euler"
    scheduler: str = "simple"
    loras: list = []
    reference_image: str = None

@app.get("/user_models")
def get_user_models():
    return {"checkpoints": [], "loras": []}

@app.post("/generate")
def generate(req: GenerateRequest):
    try:
        seed = req.seed if req.seed > 0 else random.randint(1, 2**32)
        print(f"Generating: {req.steps} steps, cfg={req.cfg}, {req.width}x{req.height}")

        # Clear cache before generation
        torch.cuda.empty_cache()
        gc.collect()

        with torch.inference_mode():
            pos = CLIPTextEncode.encode(clip, req.positive)[0]
            neg = ConditioningZeroOut.zero_out(pos)[0]
            latent = EmptyLatentImage.generate(req.width, req.height, 1)[0]
            samples = KSampler.sample(unet, seed, req.steps, req.cfg, req.sampler, req.scheduler, pos, neg, latent, 1.0)[0]
            image = VAEDecode.decode(vae, samples)[0]

        img = Image.fromarray((image.cpu().numpy().clip(0,1)*255).astype("uint8"))
        buf = io.BytesIO()
        img.save(buf, format="PNG")
        print("Done!")

        # Clear after
        torch.cuda.empty_cache()
        gc.collect()

        return {"image": base64.b64encode(buf.getvalue()).decode(), "seed": seed}
    except Exception as e:
        print(f"ERROR: {e}")
        import traceback
        traceback.print_exc()
        torch.cuda.empty_cache()
        gc.collect()
        return {"error": str(e)}

@app.get("/")
def root():
    return {"status": "ok"}

def start():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

!fuser -k 8000/tcp || true
!pkill -f ngrok || true
time.sleep(2)

!wget -q -O /tmp/ngrok.tgz https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
!tar -xzf /tmp/ngrok.tgz -C /tmp
!/tmp/ngrok config add-authtoken {NGROK_AUTHTOKEN}

threading.Thread(target=start, daemon=True).start()
print("Starting server...")
time.sleep(5)

subprocess.Popen(["/tmp/ngrok", "http", "8000"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

url = None
for _ in range(15):
    try:
        r = requests.get("http://127.0.0.1:4040/api/tunnels", timeout=2)
        data = r.json()
        for t in data.get("tunnels", []):
            if t.get("proto") == "https":
                url = t["public_url"]
                break
        if url:
            break
    except:
        pass
    time.sleep(1)

if url:
    print(f"\n{'='*50}")
    print(f"READY: {url}")
    print(f"{'='*50}")
    print("Waiting for requests...")
else:
    print("Failed to get ngrok URL")

while True:
    time.sleep(30)
    print(".", end="", flush=True)

/content/ComfyUI
Loading models...
Models loaded.
^C
Authtoken already set in configuration file: /root/.config/ngrok/ngrok.yml
Starting server...

READY: https://barrier-buccaneer-divinity.ngrok-free.dev
Waiting for requests...
...Generating: 20 steps, cfg=4.0, 1024x1024
...